# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

from skewerjax.fields.fields import ScalarField, VectorField, Fields
import skewerjax.fields.toy_model_samplers as toy_samplers
import skewerjax.utils.aux as aux

from skewerjax.optical_depth import line_profile
import skewerjax.optical_depth.optical_depth as od

from skewerjax.constants.physical_constants import phys
from skewerjax.optical_depth.voigt import Lorentzian
from skewerjax.optical_depth.voigt import Gaussian

In [ ]:
# Create cosmology object
cosmology = Cosmology(
    H0=phys.H0,
    Omega_r_0=1.0,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.0,
    Omega_k_0=0.0,
    Omega_lambda_0=0.0,
)

# Define which cosmology to use, this is set as default to 0 since skewer and cosmology objects can deal with multiple cosmologies at once, 
# but other classes such as fields and optical depth are currently only set up for a single cosmology at a time

ii_cosmology = 0

In [ ]:
wavelengths = jnp.linspace(970, 1230, int(1e6))  # in Angstroms

line_bundle = sjx.constants.lines.LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"])
line_bundle_null = sjx.constants.lines.LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])

In [ ]:
T0_tiny = jnp.finfo(jnp.dtype('float32')).tiny

T0_min = 1e1
T0_max = 1e8
T0_ref = 5e1

n0_min = 1e-10
n0_max = 1e-8
n0_ref = 1e-9

v0 = 0.0

L_box = 0.2    # in Mpc

In [ ]:
fields = Fields({
    "HI": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_ref, T0_ref, L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),               # UNITS: [km/s]
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min, n0_max, L_box)), # UNITS: [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),              # UNITS: [GHz]
    },
    "SiIII": {
        "temperature": ScalarField(toy_samplers.temperature_sampler(T0_ref, T0_ref, L_box)), # UNITS: [K]
        "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, L_box)),               # UNITS: [km/s]
        "number_density": ScalarField(toy_samplers.number_density_sampler(n0_min, n0_max, L_box)), # UNITS: [cm^-3]
        "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                # UNITS: [km/s]
        "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),              # UNITS: [GHz]
    }
})

In [ ]:
N_sk = 2

N_resolution_ppsk = 2
array_N_ppsk = jnp.geomspace(4, 42, N_resolution_ppsk).astype(jnp.int32)

F_over_F0 = []
for ii_N_ppsk, N_ppsk in enumerate(array_N_ppsk):
    _points        = aux.make_plane_points(L_box, 'x', L_box/2, N_sk, N_ppsk)
    skewer_bundle = SkewerBundle(spatial_points=_points, cosmologies=cosmology)
    optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle_null, fields)
    F_over_F0.append(jnp.exp(-optical_depth))

F_over_F0 = jnp.stack(F_over_F0, axis=0)
print(F_over_F0.shape) # (N_resolution_ppsk, N_lines, N_sk, N_lambda)

In [ ]:
sampled_temperatures = fields.temperature(skewer_bundle.quadri_vectors[ii_cosmology])[0, :, 0]
sampled_number_densities = fields.number_density(skewer_bundle.quadri_vectors[ii_cosmology])[0, :, 0]

In [ ]:
optical_depth_analytic = od.compute_optical_depth_analytical_radiation_domination_Thermal(
    wavelengths,
    line_bundle_null.lambda0_A,
    line_bundle_null.mass_amu,
    line_bundle_null.f_osc,
    sampled_temperatures,
    skewer_bundle.a0,
    skewer_bundle.x0,
    xem_Mpc=-L_box * jnp.ones(N_sk),
    xf_Mpc=jnp.zeros(N_sk),
    n0_cm3=sampled_number_densities
)

F_over_F0_analytic = jnp.exp(-optical_depth_analytic)

In [ ]:
F_over_F0_analytic.shape

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                             # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                             # (resolution N_ppsk, N_lines, N_sk, N_lambda)

colors = np.array([
    plt.cm.Reds(np.linspace(0.3, 0.9, N_resolution_ppsk)),
    plt.cm.Greens(np.linspace(0.3, 0.9, N_resolution_ppsk))
])

fig, axes = plt.subplots(
    nrows=2, ncols=len(line_bundle.lines), figsize=(5*len(line_bundle.lines), 7),  sharex='col', sharey='row', gridspec_kw=dict(height_ratios=[2, 1])
)

delta_lambda_A = 0.05   # adjust to taste

# Plot all profiles on the same axes in top panel
handles_colors = []
handles_N_ppsk = []
for i_sp, line in enumerate(line_bundle.lines):
    ax_top = axes[0, i_sp]
    ax_res = axes[1, i_sp]

    lambda0 = line.lambda0_A
    mask = (lambdas_np >= lambda0 - delta_lambda_A) & (lambdas_np <= lambda0 * (skewer_bundle.scale_factors.max() / skewer_bundle.scale_factors.min()) + delta_lambda_A)
    lam_win = lambdas_np[mask]

    ax_top.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax_top.axvline(line.lambda0_A, color="k", linestyle="-", linewidth=2., alpha=1.0)
    ax_res.axvline(line.lambda0_A, color="k", linestyle="-", linewidth=2., alpha=0.5)
    ax_top.axvline(line.lambda0_A * (skewer_bundle.scale_factors.max() / skewer_bundle.scale_factors.min()), color="k", linestyle="-", linewidth=2., alpha=1.0)
    ax_res.axvline(line.lambda0_A * (skewer_bundle.scale_factors.max() / skewer_bundle.scale_factors.min()), color="k", linestyle="-", linewidth=2., alpha=0.5)
    ax_res.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")

    for i_sk in range(N_sk):
        ax_top.plot(lam_win, F_over_F0_analytic[i_sp, i_sk, mask], color=colors[i_sk][-1], linewidth=4.0, linestyle='-', alpha=0.7)
        for i_ppsk, N_ppsk in enumerate(array_N_ppsk):
            ax_top.plot(lam_win, F_over_F0_np[i_ppsk, i_sp, i_sk, mask], color=colors[i_sk, i_ppsk], linewidth=1.5, linestyle='--')

            if i_sp == 0:
                _lbl = r"$N_\mathrm{ppsk} =$" + str(round(N_ppsk, 2))
                handles_N_ppsk.append(Line2D([0], [0], color=colors[i_sk, i_ppsk], lw=4, linestyle="-", label=_lbl))
        
        if i_sp == 0:
            _sampled_numbr_density_cm3 = fields.number_density(skewer_bundle.quadri_vectors[ii_cosmology])[i_sp, i_sk, 0]
            _lbl = rf"$n = {_sampled_numbr_density_cm3:.2g} \, \mathrm{{cm^{{-3}}}}$"
            handles_colors.append(Line2D([0], [0], color=colors[i_sk][-1], lw=4, linestyle="-", label=_lbl))

        if i_sp == 0:
            ax_res.axhline(y=0, color=colors[i_sk][-1], linestyle='-', linewidth=2.0, alpha=0.3)
        
        for i_ppsk, N_ppsk in enumerate(array_N_ppsk):
            residual = F_over_F0_np[i_ppsk, i_sp, i_sk, mask] - F_over_F0_analytic[i_sp, i_sk, mask]
            ax_res.plot(lam_win, residual, color=colors[i_sk, i_ppsk], linestyle='--', linewidth=1.5)

ax_top = axes[0, 0]
ax_res = axes[1, 0]

leg = ax_top.legend(handles=handles_colors, loc="lower right", fontsize=12, ncols=1)
ax_top.add_artist(leg)

leg = ax_top.legend(handles=handles_N_ppsk, loc="upper center", fontsize=12, ncols=len(sampled_number_densities))
ax_top.add_artist(leg)

handles_linestyles = [
    Line2D([0], [0], color="grey", lw=4, linestyle="--", label="SkewerJAX"),
    Line2D([0], [0], color="grey", lw=4, linestyle="-", label="Analytical")
]
leg = ax_top.legend(handles=handles_linestyles, loc="lower left", fontsize=12, ncols=1)
ax_top.add_artist(leg)

ax_top.set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")
ax_res.set_ylabel(r"Num. - Analytic")

fig.suptitle(r"Radiation domination." + r" $L_\mathrm{box} = $" + str(L_box) + r"$\,\mathrm{Mpc}$" + r"$.\, a_0 = $" + str(round(skewer_bundle.a0[0], 4)), fontsize=20)

fig.tight_layout()
plt.show()

In [ ]:
array_Delta_x = jnp.linspace(1e-5, 1e-1, 16)

a0 = 1.0

F_over_F0 = []
for ii_Delta_x, Delta_x in enumerate(array_Delta_x):

    fields = Fields({
        "HI": {
            "temperature": ScalarField(toy_samplers.temperature_sampler(T0_ref, T0_ref, Delta_x)), # UNITS: [K]
            "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, Delta_x)),               # UNITS: [km/s]
            "number_density": ScalarField(toy_samplers.number_density_sampler(n0_ref, n0_ref, Delta_x)), # UNITS: [cm^-3]
            "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                # UNITS: [km/s]
            "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),              # UNITS: [GHz]
        },
        "SiIII": {
            "temperature": ScalarField(toy_samplers.temperature_sampler(T0_ref, T0_ref, Delta_x)), # UNITS: [K]
            "velocity": VectorField(toy_samplers.velocity_sampler(0.0, 0.0, Delta_x)),               # UNITS: [km/s]
            "number_density": ScalarField(toy_samplers.number_density_sampler(n0_ref, n0_ref, Delta_x)), # UNITS: [cm^-3]
            "turbulence": ScalarField(toy_samplers.turbulence_sampler(b_turb=0.0)),                # UNITS: [km/s]
            "collisional": ScalarField(toy_samplers.collisional_sampler(A_coll=0.0)),              # UNITS: [GHz]
        }
    })

    _points = jnp.array([[0., Delta_x/2, Delta_x/2], [Delta_x, Delta_x/2, Delta_x/2]])[None]
    skewer_bundle = SkewerBundle(
        spatial_points=_points,
        a0=a0+phys.H0_over_c*Delta_x/2, # scale factor at which ICs are defined
        x0=Delta_x/2,                     # comoving distance at which ICs are defined
        cosmologies=cosmology
    )

    optical_depth = od.compute_optical_depth(skewer_bundle, wavelengths, line_bundle, fields)
    F_over_F0.append(jnp.exp(-optical_depth))

F_over_F0 = jnp.stack(F_over_F0, axis=0)
print(F_over_F0.shape) # (N_resolution_ppsk, N_lines, N_sk, N_lambda)

In [ ]:
mid_point_tau = od.compute_mid_point_tau(
    line_bundle,
    jnp.ones(skewer_bundle.a0.shape) * T0_ref,
    jnp.ones(skewer_bundle.a0.shape) * n0_ref,
    array_Delta_x,
    skewer_bundle.a0
)
mid_point_flux = jnp.exp(-mid_point_tau)

In [ ]:
# Convert JAX arrays to NumPy for plotting
lambdas_np = np.asarray(wavelengths)                             # (N_lambda,)
F_over_F0_np = np.asarray(F_over_F0)                             # (resolution N_ppsk, N_lines, N_sk, N_lambda)

colors = np.array([
    plt.cm.Greens(np.linspace(0.1, 0.95, len(array_Delta_x))),
    plt.cm.Blues(np.linspace(0.1, 0.95, len(array_Delta_x))),
    plt.cm.Reds(np.linspace(0.1, 0.95, len(array_Delta_x))),
])

fig, axes = plt.subplots(
    nrows=1, ncols=len(line_bundle.lines), figsize=(5*len(line_bundle.lines), 6), sharey='row'
)

delta_lambda_A = 0.025   # adjust to taste

# Plot all profiles on the same axes in top panel
handles_colors = []
i_sk = 0
for i_sp, line in enumerate(line_bundle.lines):
    ax = axes[i_sp]

    lambda0 = line.lambda0_A

    ax.set_title(rf"{line.name},  $\lambda_0 = {line.lambda0_A:.3f}\,\mathrm{{\AA}}$", fontsize=20)
    ax.axvline(line.lambda0_A, color="k", linestyle="-", linewidth=2., alpha=1.0)
    ax.set_xlabel(r"Wavelength $\lambda\ [\mathrm{\AA}]$")

    for i_Delta_x, Delta_x in enumerate(array_Delta_x):
        _Delta_x = array_Delta_x[i_Delta_x]
        _a_mid = a0 + phys.H0_over_c*_Delta_x/2

        lambdas_np_centered = lambdas_np * a0 / _a_mid
        mask = (lambdas_np_centered >= lambda0 - delta_lambda_A) & (lambdas_np_centered <= lambda0 + delta_lambda_A)
        lam_win = lambdas_np_centered[mask]

        ax.scatter(line.lambda0_A, mid_point_flux[i_sp, 0, 0, i_Delta_x], color=colors[i_sp, i_Delta_x], s=50, alpha=1.0)
        ax.plot(lam_win, F_over_F0_np[i_Delta_x, i_sp, i_sk, mask], color=colors[i_sp, i_Delta_x], linewidth=0.9, linestyle='-')

axes[0].set_ylabel(r"$F(\lambda) / F_0\, \left[\mathrm{adim}\right]$")

fig.suptitle(r"Radiation domination." + r" $L_\mathrm{box} = $" + str(L_box) + r"$\,\mathrm{Mpc}.\, a_0 = $" + str(round(skewer_bundle.a0[0], 4)) + r"$.\, n_0 = $" + str(n0_ref) + r"$\, [{\rm cm^{-3}}].\, T_0 = $" + str(T0_ref) + r"$\, [{\rm K}]$", fontsize=20)

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

for i_sp, line in enumerate(line_bundle.lines):
    ax.plot(array_Delta_x, mid_point_flux[i_sp, 0, 0, :], color=colors[i_sp, len(colors[0])-len(colors[0])//2], linewidth=2.0, linestyle='-')

ax.set_xlabel(r'$\Delta x$', fontsize=18)
ax.set_ylabel(r'$F(\lambda) / F_0$', fontsize=18)

fig.tight_layout()
plt.show()

\begin{align}
f
=
\exp -\left\lbrace
\frac{
2 n_0 \mathcal{C}_j \left\lbrace \Delta x / 2 \right\rbrace
}{
\left[ a_0 + \frac{H_0 \Delta x}{2 c} \right]^{2}
}
V \left( \frac{c}{\lambda_j} \left[ 1 - \frac{a_0}{ a_0 + \frac{H_0 \Delta x}{2 c} } \right] ;\; \sigma_j ,\, \gamma_j  \right)
\right\rbrace
\end{align}

In [ ]:
T0_min = 1e1
T0_max = 1e4

n0_min = 1e-10
n0_max = 1e-8

In [ ]:
N_T0 = 10
N_n0 = 3

T0 = jnp.geomspace(T0_min, T0_max, N_T0)
n0 = jnp.geomspace(n0_min, n0_max, N_n0)
Delta_x = jnp.linspace(0.001, 2.0, int(1e6))
a0 = 1.0
resolution_fraction = 0.1

In [ ]:
mid_point_tau = od.compute_mid_point_tau(
    line_bundle, T0, n0, Delta_x, a0
)
mid_point_flux = np.array(jnp.exp(-mid_point_tau))
Delta_x = np.array(Delta_x)

In [ ]:
colors = np.array([
    plt.cm.Greens(np.linspace(0.2, 0.95, N_T0)),
    plt.cm.Blues(np.linspace(0.2, 0.95, N_T0)),
    plt.cm.Reds(np.linspace(0.2, 0.95, N_T0)),
])

In [ ]:
fig, axes = plt.subplots(1, N_n0, figsize=(4.5*N_n0, 5), sharex='col', sharey='row')

for i_n0, _n0 in enumerate(n0):
    ax = axes[i_n0]
    for i_T0, _T0 in enumerate(T0):
        for i_sp, line in enumerate(line_bundle.lines):
            ax.plot(Delta_x, mid_point_flux[i_sp, i_T0, i_n0, :], color=colors[i_sp, i_T0], linewidth=2.0, linestyle='-')
    ax.set_title(rf"$n_0 = {_n0:.3g}\,\mathrm{{cm^{{-3}}}}$", fontsize=18)
    ax.set_xlabel(r'$\Delta x\, [\mathrm{Mpc}]$', fontsize=18)
    ax.set_xscale('log')
axes[0].set_ylabel(r'$F(\lambda) / F_0$', fontsize=18)

fig.suptitle(r"Radiation domination" + r"$.\, a_0 = $" + str(a0), fontsize=18)

fig.tight_layout()
plt.show()